# Binary benign vs malignant baseline
Prepared 14 October 2026. **Executed 14 October 2026; all three runs complete.**
Attach the original Kaggle skin-image dataset, ISIC 2019 Training Ground Truth CSV, and ISIC 2019 Training Metadata CSV. Enable a GPU.
Reconstructs and verifies the archived split, excludes AK, then trains three binary no-augmentation baselines. Test images are never evaluated.
Requires TensorFlow, pandas, NumPy and scikit-learn. A split mismatch stops execution; do not bypass that check. Save/download model archives before ending the Kaggle session.


In [ ]:
seed=50

In [23]:
#Import all necessary libraries
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from hashlib import sha256
import json, gc, zipfile
import numpy as np
import tensorflow as tf
from sklearn.metrics import (classification_report, confusion_matrix,
    accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score,
    average_precision_score)

In [ ]:
#Define input and output paths
root = Path.cwd() / "input"

out = Path.cwd().parent /'artifacts'
out.mkdir(parents=True, exist_ok=True)

In [ ]:
#Load input files
# Reconstruct audited source split; abort if membership differs from archive.
gt_file = next(root.rglob("ISIC_2019_Training_GroundTruth.csv"))
meta_file = next(root.rglob("ISIC_2019_Training_Metadata.csv"))

gt = pd.read_csv(gt_file)
meta = pd.read_csv(meta_file)
classes = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]

gt["image"] = gt["image"].str.upper()
meta["image"] = meta["image"].str.upper()
gt = gt.set_index("image")
meta = meta.drop_duplicates("image").set_index("image")

In [ ]:
#Load and reconstruct images
# Retain one copy per official image ID.
paths = {}
for p in sorted(root.rglob("*")):
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        image_id = p.stem.upper()
        if image_id in gt.index and image_id not in paths:
            paths[image_id] = str(p)

rows = []
for image_id, path in sorted(paths.items()):
    positives = [c for c in classes if gt.at[image_id, c] == 1]
    if len(positives) != 1:
        continue
    lesion = meta.at[image_id, "lesion_id"] if image_id in meta.index else None
    group = (
        f"lesion:{lesion}"
        if pd.notna(lesion) and str(lesion).strip()
        else f"image:{image_id}"
    )
    rows.append((image_id, path, positives[0], lesion, group))

df = pd.DataFrame(rows, columns=["image_id", "path", "label", "lesion_id", "group"])
assert len(df) == 1761, f"Expected 1761 images, found {len(df)}"

In [ ]:
#Stratified sampling into train/test/validation
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
trainval_idx, test_idx = next(outer.split(df, df["label"], df["group"]))
trainval, test = df.iloc[trainval_idx].copy(), df.iloc[test_idx].copy()

inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
train_idx, val_idx = next(inner.split(
    trainval, trainval["label"], trainval["group"]
))
train = trainval.iloc[train_idx].copy()
val = trainval.iloc[val_idx].copy()

train["split"], val["split"], test["split"] = "train", "val", "test"
splits = pd.concat([train, val, test], ignore_index=True)

In [ ]:
#Verify and save the previous step
assert not (set(train.group) & set(val.group))
assert not (set(train.group) & set(test.group))
assert not (set(val.group) & set(test.group))

output = Path.cwd().parent / "artifacts" / "isic2019_verified_split.csv"
splits.to_csv(output, index=False)

print(pd.crosstab(splits["label"], splits["split"], margins=True))
print("Saved:", output)

assert sha256(splits[['image_id', 'label', 'group', 'split']].sort_values("image_id").to_csv(index=False).encode()).hexdigest() == 'b6dffb92c47baf0b24fbf66432895f87e16718977bcbd655047d36febbc61f23', "Split differs from archived baseline"

In [ ]:
#Define binary labels (malignant/bening) #Why exclude AK?
# Binary task: preserve membership, exclude AK, keep original diagnoses
MAPPING = {**dict.fromkeys(['NV','BKL','DF','VASC'], 'benign'),
           **dict.fromkeys(['MEL','BCC','SCC'], 'malignant')}


excluded = splits[splits.label == 'AK'].copy()
splits = splits[splits.label.isin(MAPPING)].copy()
splits['original_label'] = splits.label
splits['label'] = splits.original_label.map(MAPPING)
splits['target'] = splits.label.map({'benign': 0, 'malignant': 1})

In [ ]:
# Verify binary dataset
original_rows = len(splits)
excluded = splits[splits.label == 'AK'].copy()
splits = splits[splits.label.isin(MAPPING)].copy()
expected_rows = original_rows - len(excluded)
assert len(splits) == expected_rows, "Unexpected number of rows in binary dataset"
assert splits.image_id.is_unique
assert splits.groupby("group").split.nunique().max() == 1

assert set(splits["split"].unique()) == {"train", "val", "test"}
assert set(splits["target"].unique()) == {0, 1}

# Verify no group overlap
train = splits[splits.split == "train"]
val = splits[splits.split == "val"]
test = splits[splits.split == "test"]

assert not (set(train.group) & set(val.group))
assert not (set(train.group) & set(test.group))
assert not (set(val.group) & set(test.group))

# Calculate binary dataset fingerprint
fingerprint = sha256(
    splits[["image_id", "label", "group", "split"]]
    .sort_values("image_id")
    .to_csv(index=False)
    .encode()
).hexdigest()

print("Binary dataset fingerprint:", fingerprint)

# Save binary dataset and fingerprint
splits.to_csv(out / "isic2019_binary_split.csv", index=False)
excluded.to_csv(out / "excluded_ak.csv", index=False)

with open(out / "dataset_fingerprint.json", "w") as f:
    json.dump({
        "seed": 50,
        "fingerprint": fingerprint,
        "number_of_rows": len(splits)
    }, f, indent=4)

print(pd.crosstab(splits["label"], splits["split"], margins=True))
print("Binary dataset and fingerprint saved.")